In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, current_date, input_file_name, lit
import datetime

STREAM_NAME   = "grid_load"
SOURCE_TABLE  = "energy_catalog.raw.src_grid_load"
BRONZE_PATH   = f"{BUCKET}/bronze/grid_load/"
CATALOG_TABLE = "energy_catalog.raw.bronze_grid_load"
batch_date    = datetime.date.today().strftime("%Y-%m-%d")
print(f"Source : {SOURCE_TABLE}")
print(f"Target : {CATALOG_TABLE}")
print(f"Date   : {batch_date}")

In [0]:
grid_schema = StructType([
    StructField("household_id",       StringType(),  True),
    StructField("grid_region",        StringType(),  True),
    StructField("substation_name",    StringType(),  True),
    StructField("feeder_line",        StringType(),  True),
    StructField("distribution_zone",  StringType(),  True),
    StructField("grid_operator",      StringType(),  True),
    StructField("grid_voltage",       DoubleType(),  True),
    StructField("grid_current",       DoubleType(),  True),
    StructField("grid_load_kw",       DoubleType(),  True),
    StructField("transformer_load",   DoubleType(),  True),
    StructField("line_loss_percent",  DoubleType(),  True),
    StructField("load_variation",     DoubleType(),  True),
    StructField("frequency_variation",DoubleType(),  True),
    StructField("grid_capacity_kw",   DoubleType(),  True),
    StructField("demand_forecast_kw", DoubleType(),  True),
    StructField("reserve_margin",     DoubleType(),  True),
])
print(f"Schema ready: {len(grid_schema.fields)} columns")

In [0]:
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("badRecordsPath", BAD_RECORDS)
    .schema(grid_schema)
    .load(RAW_PATH)
)
print(f"Rows read: {df_raw.count()}")
df_raw.show(3)

In [0]:
df_bronze = (
    df_raw
    .withColumn("_batch_date",   lit(batch_date))
    .withColumn("_ingestion_ts", current_timestamp())
    .withColumn("_source_file",  col("_metadata.file_path"))
    .withColumn("_stream_name",  lit(STREAM_NAME))
)
print(f"Total columns: {len(df_bronze.columns)}")  # expect 20

In [0]:
(df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("_batch_date")
    .save(BRONZE_PATH)
)
print("Write complete")
print(f"Rows in Delta: {spark.read.format('delta').load(BRONZE_PATH).count()}")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
    USING DELTA
    LOCATION '{BRONZE_PATH}'
""")
spark.sql(f"SELECT COUNT(*) AS total FROM {CATALOG_TABLE}").show()


In [0]:
spark.sql(f"OPTIMIZE {CATALOG_TABLE} ZORDER BY (household_id)")
print("Done. Bronze_grid_load complete.")